# 02 - EDA : Feature Engineering Temporel

## Objectif
Les fichiers `in_time` et `out_time` contiennent des horodatages bruts au format **wide**
(une colonne par jour ouvré, une ligne par employé). Ces données ne peuvent **pas** être
injectées telles quelles dans un modèle. Il faut les transformer en features agrégées
(moyenne d'heures, fréquence d'heures supplémentaires, etc.).

## Plan de ce notebook
1. Chargement des fichiers d'horaires
2. Conversion en datetime
3. Calcul du temps de travail quotidien
4. Agrégation par employé
5. Réflexion éthique

## Prérequis
- Avoir exécuté le notebook `01_eda_static_data.ipynb` et compris la structure des données.

---
## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

---
# PARTIE A — Données Temporelles

## A.1 Chargement des fichiers d'horaires

### Comprendre le format
Ces fichiers sont en format **wide** :
- Chaque **ligne** = un employé
- Chaque **colonne** = un jour ouvré (ex: `2015-01-02`, `2015-01-05`, ...)
- Chaque **cellule** = un horodatage (`2015-01-02 09:43:45`) ou `NA` si l'employé n'a pas travaillé ce jour-là

### Points de vigilance
- La **première colonne** est sans nom (vide) et contient l'identifiant employé → utiliser `index_col=0`
- Les cellules `NA` représentent des jours non travaillés (week-ends, congés, absences) → les conserver comme `NaN` volontairement

> **TODO :** Remplacez les chemins par les vôtres.

In [2]:
df_in  = pd.read_csv("../data/raw/in_time.csv",  index_col=0, na_values=["NA"])
df_out = pd.read_csv("../data/raw/out_time.csv", index_col=0, na_values=["NA"])

print(f"in_time  : {df_in.shape[0]} employés x {df_in.shape[1]} jours")
print(f"out_time : {df_out.shape[0]} employés x {df_out.shape[1]} jours")

# Vérifier que les deux fichiers ont la même structure
assert df_in.shape == df_out.shape, "Les deux fichiers n'ont pas la même dimension !"
assert (df_in.index == df_out.index).all(), "Les index employés ne correspondent pas !"
assert (df_in.columns == df_out.columns).all(), "Les colonnes dates ne correspondent pas !"

in_time  : 4410 employés x 261 jours
out_time : 4410 employés x 261 jours


## A.2 Conversion en datetime

### Pourquoi c'est nécessaire
À ce stade, les cellules sont de type `object` (chaînes de caractères).
Pour calculer des durées, pandas a besoin du type `datetime64`.

### Approche vectorisée
On utilise `pd.to_datetime()` avec `errors='coerce'` pour que les valeurs non-parsables
deviennent `NaT` (Not a Time) plutôt que lever une exception.

**Attention :** appliquer `pd.to_datetime` colonne par colonne via `.apply()` est acceptable ici
car l'opération est **par colonne**, ce n'est pas une boucle Python sur chaque cellule.

In [3]:
df_in  = df_in.apply(pd.to_datetime, errors="coerce")
df_out = df_out.apply(pd.to_datetime, errors="coerce")

# Vérification rapide du type
print(f"Type d'une colonne in_time  : {df_in.iloc[:, 0].dtype}")
print(f"Type d'une colonne out_time : {df_out.iloc[:, 0].dtype}")

Type d'une colonne in_time  : datetime64[s]
Type d'une colonne out_time : datetime64[s]


## A.3 Calcul du temps de travail quotidien

### Le concept
La soustraction de deux DataFrames `datetime64` produit un DataFrame de `timedelta64`.
On peut ensuite convertir en heures décimales en divisant par `pd.Timedelta(hours=1)`.

C'est une opération **entièrement vectorisée** — pas de boucle nécessaire.

In [4]:
# Soustraction vectorisée : chaque cellule = durée travaillée ce jour
df_hours = (df_out - df_in) / pd.Timedelta(hours=1)

print(f"Dimensions : {df_hours.shape}")
print(f"\nAperçu (employé 1, 5 premiers jours) :")
print(df_hours.iloc[0, :5])

Dimensions : (4410, 261)

Aperçu (employé 1, 5 premiers jours) :
2015-01-01         NaN
2015-01-02    7.208333
2015-01-05    7.189722
2015-01-06    7.410833
2015-01-07    7.006667
Name: 1, dtype: float64


## A.4 Agrégation par employé

### Pourquoi agréger ?
Un modèle de classification attend **une ligne par employé** avec des features numériques.
Les 261 colonnes de durées quotidiennes doivent être résumées en quelques indicateurs.

### Features suggérées
Réfléchissez aux agrégations qui ont du **sens métier** pour prédire l'attrition :
- Moyenne d'heures par jour travaillé
- Écart-type (régularité vs. amplitude)
- Nombre de jours travaillés sur l'année
- Proportion de jours avec heures supplémentaires (> seuil à définir)

> **TODO :** Choisissez vos agrégations et le seuil d'heures supplémentaires.

In [5]:
OVERTIME_THRESHOLD = 8  # TODO : ajustez ce seuil selon votre analyse

df_time_features = pd.DataFrame(index=df_hours.index)

df_time_features["avg_hours_per_day"]   = df_hours.mean(axis=1)
df_time_features["std_hours"]           = df_hours.std(axis=1)
df_time_features["total_days_worked"]   = df_hours.notna().sum(axis=1)
df_time_features["overtime_ratio"]      = (
    (df_hours > OVERTIME_THRESHOLD).sum(axis=1) / df_hours.notna().sum(axis=1)
)

print(df_time_features.describe().round(2))

       avg_hours_per_day  std_hours  total_days_worked  overtime_ratio
count            4410.00    4410.00            4410.00         4410.00
mean                7.70       0.30             236.27            0.32
std                 1.34       0.01               5.50            0.42
min                 5.95       0.25             225.00            0.00
25%                 6.67       0.29             232.00            0.00
50%                 7.41       0.30             236.00            0.03
75%                 8.37       0.31             241.00            0.88
max                11.03       0.34             248.00            1.00


### Observations — Données temporelles

**Seuil d'heures supplémentaires :**
- Seuil choisi : [...] heures → justification : [...]
- Après avoir regardé la distribution de `avg_hours_per_day`, ce seuil est-il pertinent ? [...]

**Employés sans jours travaillés :**
- Y a-t-il des employés avec `total_days_worked = 0` ? [...] → si oui, que signifient-ils ? [...]

**Régularité vs amplitude :**
- Que nous dit `std_hours` ? Un écart-type élevé signifie [...] → est-ce corrélé à l'attrition ? (hypothèse à vérifier en Sprint 2)

**Features temporelles retenues pour la modélisation :**
- [...] (listez les features que vous garderez et pourquoi)

---
## 5. Réflexion Éthique

Dans ce notebook, nous avons calculé pour chaque employé sa **moyenne d'heures travaillées**,
sa **régularité** et sa **proportion d'heures supplémentaires** à partir de pointages horodatés.

Ces données de pointage sont des **données de surveillance** au sens du RGPD.
Les utiliser pour prédire le départ d'un employé transforme un outil de gestion du temps
en un **instrument de profilage comportemental**.

### Question
Si un employé découvre que ses horaires de badge servent à alimenter un modèle prédictif
de départ, cela constitue-t-il un **détournement de finalité** au sens du RGPD ?

Parmi les 7 exigences de l'UE pour une IA de confiance, laquelle impose que les individus
sachent **quand** et **comment** leurs données sont utilisées par un système automatisé ?

> *Notez votre réflexion ici avant de poursuivre vers la modélisation.*

### Questions à vous poser :
- Y a-t-il des colonnes avec des valeurs **physiquement impossibles** (âge < 0, heures < 0) ?
- Certaines colonnes ont-elles une variance quasi nulle (`std ≈ 0`) ? Si oui, apportent-elles de l'information au modèle ?
- Les outliers extrêmes sont-ils des **erreurs de saisie** ou des **cas métier légitimes** ?